## Setup do Ambiente

- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    count,
    min,
    round,
    max,
    sum,
    avg,
    lit,
    desc,
    struct,
    concat_ws,
    collect_list,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when,
    to_date,    
    current_timestamp
)
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.utils import AnalysisException

#### Definição de Variáveis Globais

- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Funções Úteis

### Função `table_check`

Esta função verifica se uma tabela existe em um banco de dados específico e se ela contém dados.

In [0]:
def table_check(table_name, db_name):
    """
    Verifica se uma tabela existe e possui dados em um banco de dados especificado.

    Args:
        table_name (str): Nome da tabela a ser verificada.
        db_name (str): Nome do banco de dados onde a tabela está localizada.

    Returns:
        bool: True se a tabela existe e possui dados, False caso contrário.

    Raises:
        ValueError: Se a tabela existe mas está vazia.
    """
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
        if spark.table(f"{db_name}.{table_name}").count() == 0:
            raise ValueError(f"Tabela {db_name}.{table_name} existe mas está vazia.")
        return True
    return False

### Função `save_table_gold` 
Salva uma tabela a partir do catálogo (`catalogo`) e camada (`gold_db_name`) especificados, em formato Delta.
Adiciona a coluna `data_criacao_gold`

In [0]:
def save_table_gold(table_name: str, df, process_col=True):
    """
    Salva um DataFrame como tabela Delta na camada gold.

    Args:
        table_name (str): Nome da tabela a ser criada ou sobrescrita na camada gold.
        df (DataFrame): DataFrame Spark que será salvo como tabela Delta.
        process_col(Bool): Adicionar uma coluna com o current timestamp.
    """
    table_path = f"{catalogo}.{gold_db_name}.{table_name}"

    try:
        old_schema = spark.table(table_path).schema.simpleString()
    except AnalysisException:
        old_schema = None

    if process_col == True:
        df = df.withColumn("data_criacao_gold", current_timestamp())
    
    try:
        df.write \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(table_path)
    except Exception as ex:
        print(f"Erro ao salvar a tabela {table_path}: {ex}")
        return
    
    new_schema = spark.table(table_path).schema.simpleString()

    if old_schema is None:
        print(f"Tabela {table_path} criada pela primeira vez.")
    elif old_schema != new_schema:
        print(f"Esquema da tabela {table_path} foi alterado.\n")
        print("Schema anterior:")
        print(old_schema)
        print("\nNovo schema:")
        print(new_schema)
    else:
        print(f"Tabela salva com sucesso: {table_path}")

### Função `read_table`
Lê uma tabela do Databricks a partir do catálogo e camada especificados (`silver` ou `gold`), retornando um DataFrame Spark correspondente.

In [0]:
def read_table(nome_tabela: str, camada:str='silver'):
    """
    Lê uma tabela Delta da camada especificada.

    Args:
        nome_tabela (str): Nome da tabela a ser lida.
        camada (str, optional): Camada de origem da tabela ('silver' ou 'gold'). Default é 'bronze'.

    Returns:
        DataFrame: DataFrame Spark da tabela lida.

    Raises:
        ValueError: Se a camada não for 'silver' ou 'gold'.
        ValueError: Se a tabela não existir ou estiver vazia.
    """
    db_map = {
        'silver': silver_db_name,
        'gold': gold_db_name
    }

    db_nome = db_map.get(camada.lower())
    
    if not db_nome:
        raise ValueError("Camada deve ser 'silver' ou 'gold'")
    
    if not table_check(nome_tabela, db_nome):
        raise ValueError(f"Tabela {db_nome}.{nome_tabela} não existe.")
    return spark.table(f"{catalogo}.{db_nome}.{nome_tabela}")

### Função `describe_table`
Exibe o schema, contagem de linhas com valores nulos e 5 linhas da tabela.

In [0]:
def describe_table(df):
    """
    Exibe o schema, linhas com valores nulos e as primeiras 5 linhas do DataFrame usando display.

    Args:
        df (DataFrame): DataFrame Spark a ser exibido.
    """
    df.printSchema()
    print(f"Total de linhas: {df.count()}")
    null_count = df.filter(
        reduce(lambda a, b: a | b, [F.col(c).isNull() for c in df.columns])
    ).count()
    print(f"Linhas com valores nulos: {null_count}")
    display(df.limit(5))

## Criação da Tabela `dm_tempo`

Colunas da tabela `dm_tempo`:

- `sk_tempo`
- `ano`
- `trimestre`
- `mes`
- `semana_do_ano`
- `dia`
- `dia_da_semana_num`
- `dia_da_semana_nome`
- `mes_nome`
- `eh_fim_de_semana`

In [0]:
df_ft_chamados_hora = read_table("ft_chamados_hora", "silver")

min_max_tempo = df_ft_chamados_hora.agg(
    F.min("hora_abertura_chamado").alias("min_tempo"),
    F.max("hora_finalizacao_atendimento").alias("max_tempo")
)

min_max_tempo.withColumn("min_data", to_date("min_tempo")).withColumn("max_data", to_date("max_tempo")).select("min_data", "max_data").display()

Datas usadas no dataset: 
- 2025-01-01 à 2025-06-30

In [0]:
data_inicio = '2025-01-01'
data_fim = '2025-06-30'

df_datas = (
    spark.createDataFrame([(data_inicio, data_fim)], ['data_inicio', 'data_fim'])
    .select(explode(sequence(col('data_inicio').cast(DateType()), col('data_fim').cast(DateType()))).alias('sk_tempo'))
    .withColumn('ano', year(col('sk_tempo')))
    .withColumn('trimestre', quarter(col('sk_tempo')))
    .withColumn('mes', month(col('sk_tempo')))
    .withColumn('semana_do_ano', weekofyear(col('sk_tempo')))
    .withColumn('dia', dayofmonth(col('sk_tempo')))
    .withColumn('dia_da_semana_num', dayofweek(col('sk_tempo')))
    .withColumn('dia_da_semana_nome', 
        when(col('dia_da_semana_num') == 1, 'Domingo')
        .when(col('dia_da_semana_num') == 2, 'Segunda-feira')
        .when(col('dia_da_semana_num') == 3, 'Terça-feira')
        .when(col('dia_da_semana_num') == 4, 'Quarta-feira')
        .when(col('dia_da_semana_num') == 5, 'Quinta-feira')
        .when(col('dia_da_semana_num') == 6, 'Sexta-feira')
        .when(col('dia_da_semana_num') == 7, 'Sabado')
    )
    .withColumn('mes_nome',
        when(col('mes') == 1, 'Janeiro')
        .when(col('mes') == 2, 'Fevereiro')
        .when(col('mes') == 3, 'Março')
        .when(col('mes') == 4, 'Abril')
        .when(col('mes') == 5, 'Maio')
        .when(col('mes') == 6, 'Junho')
        .when(col('mes') == 7, 'Julho')
        .when(col('mes') == 8, 'Agosto')
        .when(col('mes') == 9, 'Setembro')
        .when(col('mes') == 10, 'Outubro')
        .when(col('mes') == 11, 'Novembro')
        .when(col('mes') == 12, 'Dezembro')
    )
    .withColumn('eh_fim_de_semana', when(col('dia_da_semana_num').isin([1,7]), 'Sim').otherwise('Não'))
)

describe_table(df_datas)

save_table_gold("dm_tempo", df_datas)

#### Join com `ft_chamados_geral`

In [0]:
df_ft_chamados_geral = read_table("ft_chamados_geral", "silver")

df_dm_tempo = read_table("dm_tempo", "gold")

df_join = df_ft_chamados_geral.join(
    df_dm_tempo,
    to_date(col("hora_abertura_chamado")) == df_dm_tempo["sk_tempo"],
    "inner"
)

save_table_gold("ft_chamados_tempo", df_join)

describe_table(df_join)

## Criação da Tabela `ft_chamados`

Colunas da tabela `ft_chamados`:

- `id_chamado`
- `id_cliente`
- `id_atendente`
- `motivo`
- `canal`
- `resolvido`
- `nota_atendimento`
- `categoria_nota`
- `status_canal`
- `valor_custo`

In [0]:
df_chamados_geral = spark.table(f'{catalogo}.{silver_db_name}.ft_chamados_geral')
display(df_chamados_geral.limit(20))

In [0]:
print(f"Colunas de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
print(f"{df_chamados_geral.columns}\n")
print(f"Schema de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
df_chamados_geral.printSchema()

In [0]:
df_gold_ft_chamados = df_chamados_geral.select(
    col('id_chamado'),
    col('id_cliente'),
    when(col('id_atendente') == -1, None).otherwise(col('id_atendente')).alias('id_atendente'),
    when(col('nome_atendente').isNull(), "Atendimento Não Humano").otherwise(col('nome_atendente')).alias('nome_atendente'),
    col('motivo'),
    col('hora_inicio_atendimento'),
    col('hora_finalizacao_atendimento'),
    col('canal'),
    col('status_canal'),
    col('resolvido'),
    col('nota_atendimento'),
    col('categoria_nota'),
    col('valor_custo')
)
df_gold_ft_chamados.limit(15).display()
df_gold_ft_chamados.printSchema()

In [0]:
df_tratado = df_gold_ft_chamados.withColumn("ts_inicio", col("hora_inicio_atendimento")) \
                   .withColumn("ts_fim", col("hora_finalizacao_atendimento")) 

GAP_LIMIT_SECONDS = 7 * 24 * 60 * 60  # 7 dias

w_cliente_motivo = Window.partitionBy("id_cliente", "motivo").orderBy(col("ts_inicio"))

Cria uma janela de particionamento para análise temporal
- Agrupa por: cliente + motivo do chamado
- Ordena por: data/hora de início (do mais antigo ao mais recente).Isso permite comparar chamados consecutivos do mesmo cliente sobre o mesmo assunto

In [0]:
df_sessao = df_tratado.withColumn(
    "ts_fim_anterior", F.lag("ts_fim").over(w_cliente_motivo)
).withColumn(
    "segundos_desde_ultimo_contato", 
    col("ts_inicio").cast("long") - col("ts_fim_anterior").cast("long")
).withColumn(
    "nova_jornada_flag",
    when(
        (col("segundos_desde_ultimo_contato").isNull()) | 
        (col("segundos_desde_ultimo_contato") > GAP_LIMIT_SECONDS), 
        1
    ).otherwise(0)
)
df_sessao.filter(col("canal") == "Chatbot").limit(10).display()

- Busca o timestamp de finalização do chamado ANTERIOR
-   Usa lag() para "olhar para trás" na janela ordenada
-   Retorna NULL na primeira linha de cada partição (não há anterior)
-   Subtrai: (quando este chamado começou) - (quando o anterior terminou)
-   Conversão para "long" 
-    Resultado NULL indica que é o primeiro chamado da partição
-  Flag binária que marca o início de uma nova jornada
-   Valor 1: É uma nova jornada (primeiro contato OU gap > 7 dias)
-   Valor 0: Continua na mesma jornada (gap <= 7 dias)

In [0]:
ft_jornada_atendimento = df_sessao.groupBy("id_cliente", "motivo").agg(
    
    min("ts_inicio").alias("data_inicio_jornada"),
    max("ts_fim").alias("data_fim_jornada"),
    count("id_chamado").alias("qtd_tentativas"),
    
    round(
        (max("ts_fim").cast("long") - min("ts_inicio").cast("long")) / 1000 / 60, 2
    ).alias("duracao_jornada_minutos"),

    round(sum("valor_custo"), 2).alias("custo_total_jornada"),
    
    max(when(col("id_atendente") != -1, 1).otherwise(0)).alias("flg_passou_humano"),
    
    # Pega o status e categoria_nota do chamado mais recente (maior data de inicio)
    max(struct("ts_inicio", "resolvido"))["resolvido"].alias("status_final_resolucao"),
    max(struct("ts_inicio", "categoria_nota"))["categoria_nota"].alias("satisfacao_cliente"),
    
    concat_ws(" -> ", collect_list("canal")).alias("caminho_canais")
)

ft_jornada_atendimento.orderBy("data_inicio_jornada").limit(10).display()

In [0]:
df_gold_ft_chamados.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.ft_chamados')
ft_jornada_atendimento.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.ft_jornada_atendimento')

### Views

In [0]:
df_jornada = spark.read.table('ft_jornada_atendimento')
df_jornada.printSchema()

#### View Operacional por Tipo de Atendimento
- **Objetivo**:  comparar jornadas que passaram por atendimento humano versus jornadas atendidas apenas por bot, divididas por status final de resolução.
- Passos: 
  -  Cria uma coluna categórica baseada na flag `flg_passou_humano`
  - Agrupa por tipo de atendimento e status de resolução
- Resultado:
  - `total_tipo:` Total de jornadas dentro de cada tipo de atendimento
  -` perc_status:` Percentual que cada status representa dentro do seu tipo
| Métrica | Descrição | Fórmula |
|---------|-----------|---------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) |
| `custo_total` | Custo acumulado | SUM(custo_total_jornada) |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) |
| `tentativas_max` | Máximo de tentativas | MAX(qtd_tentativas) |


In [0]:
df_operacional = df_jornada \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .groupBy("tipo_atendimento", "status_final_resolucao") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(sum("custo_total_jornada"), 2).alias("custo_total"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(max("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("total_tipo", sum("qtd_jornadas").over(Window.partitionBy("tipo_atendimento"))) \
    .withColumn("perc_status", round((col("qtd_jornadas") / col("total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .withColumn("custo_total_tipo", round(sum("custo_total").over(Window.partitionBy("tipo_atendimento")), 2)) \
    .withColumn("perc_custo", round((col("custo_total") / col("custo_total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("tipo_atendimento", desc("qtd_jornadas")) \
    .withColumn("qtd_resolvidos_tipo", 
                sum(when(col("status_final_resolucao") == "Sim", col("qtd_jornadas")).otherwise(0))
                 .over(Window.partitionBy("tipo_atendimento"))) \
    .withColumn("taxa_resolucao_tipo", 
                round((col("qtd_resolvidos_tipo") / col("total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("tipo_atendimento", desc("qtd_jornadas"))

df_operacional.display()

#### Análise Detalhada por Status de Resolução e Motivo

- **Objetivo:**
Analisar a distribuição de jornadas por **status de resolução** e **motivo de contato**, permitindo identificar quais motivos têm maior taxa de resolução, custo e necessidade de intervenção humana.
- **Passos:**
Agrupar por status de resolução e motivos.

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume absoluto dessa combinação |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Taxa de escalonamento para humano |
| `perc_dentro_status` | %  cada motivo representa dentro do seu status | Window function soma todas as jornadas do mesmo status | "Do total de jornadas resolvidas, 30% foram sobre Boleto"|
| `perc_total` | Calcula quanto cada combinação status+motivo representa do total geral|  Window.partitionBy(F.lit(1)) cria uma janela sobre toda a tabela | "Esta combinação representa 6% de todas as jornadas" |


In [0]:
df_status_motivo = df_jornada \
    .groupBy("status_final_resolucao", "motivo") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(avg("qtd_tentativas"), 2).alias("tentativas_media"),
        round((sum("flg_passou_humano") / count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("perc_dentro_status", round((col("qtd_jornadas") /  sum("qtd_jornadas").over(Window.partitionBy("status_final_resolucao"))* 100).cast("decimal(5,2)"), 2)) \
    .withColumn("perc_total", 
    round((col("qtd_jornadas") / 
             sum("qtd_jornadas").over(Window.partitionBy(lit(1))) * 100)
            .cast("decimal(5,2)"), 2)) \
    .orderBy("status_final_resolucao", desc("qtd_jornadas"))

df_status_motivo.display()

#### Análise de Satisfação do Cliente por Motivo

- **Objetivo:**
Analisar como os clientes avaliam sua experiência em cada **motivo de contato**, correlacionando satisfação com métricas operacionais como custo, duração e tipo de atendimento.

- **Passos:**
1. Filtrar apenas jornadas onde o cliente avaliou (`satisfacao_cliente IS NOT NULL`)
2. Agrupar por motivo e satisfação do cliente
3. Calcular métricas operacionais por combinação motivo + satisfação
4. Adicionar contexto percentual dentro de cada motivo

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_avaliacoes` | Total de avaliações | COUNT(*) | Volume de feedback nessa combinação |
| `custo_medio` | Custo médio | AVG(custo_total_jornada) | Correlacionar custo × satisfação |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Ver se tempo impacta satisfação |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Verificar se humano melhora satisfação |
| `total_motivo` | Total de avaliações do motivo | Window function soma todas avaliações do mesmo motivo | Base para cálculo percentual |
| `perc_satisfacao` | % que cada nota representa no motivo | (qtd_avaliacoes / total_motivo) × 100 | "70% dos clientes que avaliaram Boleto ficaram satisfeitos" |


In [0]:
df_satisfacao_motivo = df_jornada \
    .filter(col("satisfacao_cliente").isNotNull()) \
    .groupBy("motivo", "satisfacao_cliente") \
    .agg(
        count("*").alias("qtd_avaliacoes"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round((sum("flg_passou_humano") / count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("total_motivo", sum("qtd_avaliacoes").over(Window.partitionBy("motivo"))) \
    .withColumn("perc_satisfacao", round((col("qtd_avaliacoes") / col("total_motivo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("motivo", "satisfacao_cliente")

df_satisfacao_motivo.display()

#### Análise de Eficiência por Tipo de Atendimento

- **Objetivo:**
Comparar a **eficiência operacional** de jornadas atendidas apenas por bot versus jornadas que passaram por atendimento humano, segmentadas por motivo de contato.

- **Passos:**
1. Agrupar por motivo e flag de atendimento humano
2. Calcular métricas de performance (duração, custo, tentativas)
3. Calcular custo por minuto para medir eficiência
4. Classificar tipo de atendimento para melhor legibilidade
5. Ordenar por motivo e tipo para facilitar comparação

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume em cada tipo de atendimento |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `tipo_atendimento` | Classificação do atendimento | "Humano + Bot" ou "Apenas Bot" | Facilita leitura e análise |

In [0]:
df_eficiencia = df_jornada \
    .groupBy("motivo", "flg_passou_humano") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .select("motivo", "tipo_atendimento", "qtd_jornadas", "duracao_media_min", "custo_medio", "tentativas_media") \
    .orderBy("motivo", "tipo_atendimento")

df_eficiencia.display()

#### Análise de Satisfação por Tipo de Atendimento

- **Objetivo:**
Mapear a distribuição das **notas de satisfação** dos clientes, comparando a experiência em canais 100% digitais ("Apenas Bot") versus interações híbridas ("Humano + Bot"), detalhado por motivo de contato.

- **Passos:**
1. Filtrar registros para garantir que apenas jornadas avaliadas sejam processadas.
2. Agrupar os dados por motivo, tipo de atendimento e nota atribuída.
3. Calcular métricas absolutas (quantidade de notas e duração média).
4. Utilizar **Window Function** para calcular o total de avaliações por grupo, permitindo o cálculo percentual de cada nota (share).
5. Rotular o tipo de atendimento e ordenar para análise visual.

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `satisfacao_cliente` | Nota da avaliação | Coluna original | A nota dada pelo cliente (ex: CSAT ou NPS) |
| `qtd_avaliacoes` | Volume de notas | COUNT(*) | Quantas pessoas deram especificamente essa nota |
| `perc_satisfacao` | % Representatividade | (Qtd Nota / Total do Tipo) * 100 | Qual a porcentagem de clientes que deram essa nota dentro desse canal |
| `duracao_media_min` | Tempo médio | AVG(duracao_jornada_minutos) | Relação entre o tempo de atendimento e a satisfação |
| `tipo_atendimento` | Canal utilizado | Case When... | Segmentação entre Bot Puro ou Transbordo Humano |

In [0]:
window_spec = Window.partitionBy("motivo", "flg_passou_humano")

df_eficiencia_satisfacao = df_jornada \
    .filter(col("satisfacao_cliente").isNotNull()) \
    .groupBy("motivo", "flg_passou_humano", "satisfacao_cliente") \
    .agg(
        count("*").alias("qtd_avaliacoes"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min")
    ) \
    .withColumn("total_por_tipo", F.sum("qtd_avaliacoes").over(window_spec)) \
    .withColumn("perc_satisfacao", 
                round((col("qtd_avaliacoes") / col("total_por_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .select("motivo", "tipo_atendimento", "satisfacao_cliente", "qtd_avaliacoes", "perc_satisfacao", "duracao_media_min") \
    .orderBy("motivo", "tipo_atendimento", "satisfacao_cliente")

df_eficiencia_satisfacao.display()

In [0]:
df_operacional.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_operacional')
df_status_motivo.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_status_motivo')
df_satisfacao_motivo.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_satisfacao_motivo')
df_eficiencia.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_eficiencia')
df_eficiencia_satisfacao.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_eficiencia_satisfacao')